# 8.2 Twitter API

- <a href='#8.1.'>8.1. Generate Twitter API Key</a>
- <a href='#8.2.'>8.2. Tweepy </a> 
     - <a href='#8.2.1.'>8.2.1. Get tweets in a date range </a> 
- <a href='#8.3.'>8.3. Request tweet </a>  
     - <a href='#8.3.1.'>8.3.1. Getting ID of username - jmwooldridge</a> 
     - <a href='#8.3.2.'>Getting Tweets Timeline - jmwooldridge</a> 
- <a href='#8.4.'>8.4. References</a>  

In [ ]:
# For sending GET requests from the API
import requests
# For saving access tokens and for file management when creating and adding to the dataset
import os
# For dealing with json responses we receive from the API
import json
# For displaying the data after
import pandas as pd
# For saving the response data in CSV format
import csv
# For parsing the dates received from twitter in readable formats
import datetime
import dateutil.parser
import unicodedata
#To add wait time between requests
import time

## <a id='8.1.'>8.1. Generate Twitter API Key</a>

All you need to do is create a project and connect an App through the developer portal and we are set to go!

1. Go to the developer [portal dashboard](https://developer.twitter.com/en/portal/dashboard).
2. Sign in with your developer account.
3. Create a new project, give it a name, a use-case based on the goal you want to achieve, and a description.
<img src="../_images/1_twitter.png" />

4. Assuming this is your first time, choose ‘create a new App instead’ and give your App a name in order to create a new App.
<img src="../_images/2_twitter.png" />

5. If everything is successful, you should be able to see this page containing your keys and tokens, we will use one of these to access the API.
<img src="../_images/3_twitter.png" />

### <a id='8.2.'>8.2. Tweepy </a> 

In [ ]:
!pip install tweepy

In [ ]:
import tweepy
consumer_key = "Rr3sWjlZW6bdDlL2pRf1pbtia"
consumer_secret = "mxhZpvR8BX96Sd347oCcNRDDjpdwHVxItwWE28ahKpiuQgvRe4"
auth = tweepy.OAuthHandler( consumer_key, consumer_secret )

access_token = "1270850339223343105-DLleC3HxuR45fyVJOJHsQ5YEaXv8kO"
access_token_secret = "lNJ1iQVmHJIw0QZHaC19svWk2GbCthmzhsk4jXaVStmcM"
auth.set_access_token( access_token, access_token_secret )

api = tweepy.API(auth)

Get Timeline of user

In [ ]:
# Get Timeline
public_tweets = api.home_timeline()
for tweet in public_tweets:
    print( tweet.text )

Getting tweets from [Judea Pearl](https://twitter.com/yudapearl).

In [ ]:
userID = "yudapearl"

In [ ]:
# Set uttils
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)

# Set API
api = tweepy.API(auth)

# Get Tweets
tweets_bi = api.user_timeline(screen_name = userID, 
                           # 200 is the maximum allowed count
                           count = 100,
                           include_rts = False,
                           # Necessary to keep full_text 
                           # otherwise only the first 140 words are extracted
                           tweet_mode = 'extended'
                           )

In [ ]:
print( tweets_bi[0].id )
print( tweets_bi[0].created_at )
print( tweets_bi[0].full_text )

In [ ]:
for info in tweets_bi[:3]:
     print("ID: {}".format(info.id))
     print(info.created_at)
     print(info.full_text)
     print("\n")

In [ ]:
tweets_bi[0].full_text

In [ ]:
from pandas import DataFrame
outtweets = [[tweet.id_str, 
              tweet.created_at, 
              tweet.favorite_count, 
              tweet.retweet_count, 
              tweet.full_text.encode("utf-8").decode("utf-8")] 
             for idx,tweet in enumerate(tweets_bi)]
df = DataFrame(outtweets,columns=["id","created_at","favorite_count","retweet_count", "text"])
df.to_csv( f'{userID}_tweets.csv', index = False )
df.head(3)

In [ ]:
df

## <a id ='8.3.'>8.3. Request tweet </a>

In [ ]:

# For sending GET requests from the API
import requests
# For saving access tokens and for file management when creating and adding to the dataset
import os
# For dealing with json responses we receive from the API
import json
# For displaying the data after
import pandas as pd
# For saving the response data in CSV format
import csv
# For parsing the dates received from twitter in readable formats
import datetime
import dateutil.parser
import unicodedata
#To add wait time between requests
import time

## <a id ='8.3.1.'>8.3.1. Getting ID of username - [jmwooldridge](https://twitter.com/jmwooldridge) </a> 

In [ ]:
class request_id:
    
    def __init__( self, token):
        self.token = token
    
    def create_url( self, user_name ):
        # Specify the usernames that you want to lookup below
        # You can enter up to 100 comma-separated values.
        usernames = f"usernames={ user_name }"
        user_fields = "user.fields=description,created_at"
        # User fields are adjustable, options include:
        # created_at, description, entities, id, location, name,
        # pinned_tweet_id, profile_image_url, protected,
        # public_metrics, url, username, verified, and withheld
        url = "https://api.twitter.com/2/users/by?{}&{}".format(usernames, user_fields)
        self.url = url


    def bearer_oauth( self, r ):
        """
        Method required by bearer token authentication.
        """

        r.headers["Authorization"] = f"Bearer {self.token}"
        r.headers["User-Agent"] = "v2UserLookupPython"
        
        self.r = r
        return r


    def connect_to_endpoint( self ):
        url = self.url
        response = requests.request("GET", url, auth= self.bearer_oauth,)
        print(response.status_code)
        if response.status_code != 200:
            raise Exception(
                "Request returned an error: {} {}".format(
                    response.status_code, response.text
                )
            )
            
        return response.json()

Token

In [ ]:
Bearer_Token = "AAAAAAAAAAAAAAAAAAAAAHqDOQEAAAAA13Ox6r2pGXSajjvKhZUT%2BksIvnk%3DhdyhLtNoNYKm50YORIqKDBakGh5JTvjl828R3mToJzFmWjb1CJ"
get_id_woold = request_id( Bearer_Token )

Username

In [ ]:
get_id_woold.create_url( 'jmwooldridge' )

result = get_id_woold.connect_to_endpoint()

Getting Data

In [ ]:
id_woold = result['data'][0]['id']
id_woold

## <a id = '8.3.2.'>8.3.2. [Getting Tweets Timeline](https://developer.twitter.com/en/docs/twitter-api/tweets/timelines/introduction) - [jmwooldridge](https://twitter.com/jmwooldridge) </a> 

The endpoint can return the 3,200 most recent Tweets, Retweets, replies and Quote Tweets posted by the user. <br>
The user Tweet timeline also supports the ability to specify `start_time` and `end_time` parameters to receive Tweets that were created within a certain window of time. 

In [ ]:
def create_url(  user_id ):
    # Replace with user ID below
    return "https://api.twitter.com/2/users/{}/tweets".format(user_id)


def get_params():
    # Tweet fields are adjustable.
    # Options include:
    # attachments, author_id, context_annotations,
    # conversation_id, created_at, entities, geo, id,
    # in_reply_to_user_id, lang, non_public_metrics, organic_metrics,
    # possibly_sensitive, promoted_metrics, public_metrics, referenced_tweets,
    # source, text, and withheld
    return {"tweet.fields": "created_at", "max_results" : 100 }


def bearer_oauth(r):
    """
    Method required by bearer token authentication.
    """
    r.headers["Authorization"] = f"Bearer {bearer_token}"
    r.headers["User-Agent"] = "v2UserTweetsPython"
    return r


def connect_to_endpoint(url, params, next_token = None ):
    params['pagination_token'] = next_token
    response = requests.request("GET", url, auth=bearer_oauth, params=params)
    print(response.status_code)
    if response.status_code != 200:
        raise Exception(
            "Request returned an error: {} {}".format(
                response.status_code, response.text
            )
        )
    return response.json()

In [ ]:
bearer_token = 'AAAAAAAAAAAAAAAAAAAAAHqDOQEAAAAA13Ox6r2pGXSajjvKhZUT%2BksIvnk%3DhdyhLtNoNYKm50YORIqKDBakGh5JTvjl828R3mToJzFmWjb1CJ'

In [ ]:
url = create_url( id_woold )
param = get_params()
result = connect_to_endpoint( url, param)

In [ ]:
result['meta']

In [ ]:
data = result['data']

In [ ]:
next_token = 'next_token' in result['meta'].keys()
page = 1
while next_token == True:
    try:
        restult = connect_to_endpoint( url, param, result['meta']['next_token'] )
        data = data + result['data']
    except:
        next_token = False
    
    if page > 10:
        next_token = False
    
    page = page + 1

In [ ]:
df = pd.DataFrame( data )

In [ ]:
df

We are allowed to request only 3200 tweets.

## <a id='8.4.'>8.4. References</a>  

https://towardsdatascience.com/an-extensive-guide-to-collecting-tweets-from-twitter-api-v2-for-academic-research-using-python-3-518fcb71df2a

https://fairyonice.github.io/extract-someones-tweet-using-tweepy.html